In [ ]:
import pandas as pd
import numpy as np
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import os
import PIL

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [ ]:
with ZipFile('archive.zip', 'r') as zip:
    zip.extractall()

In [ ]:
path_images = "/content/brain_tumor_dataset"
BATCH_SIZE = 32
IMG_SIZE = (180,  180)
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
epochs = 30

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  path_images,
  validation_split=0.2,
  subset="training",
  seed=SEED,
  image_size=(IMG_SIZE),
  batch_size=BATCH_SIZE)

Found 253 files belonging to 2 classes.
Using 203 files for training.


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  path_images,
  validation_split=0.2,
  subset="validation",
  seed=SEED,
  image_size=(IMG_SIZE),
  batch_size=BATCH_SIZE)

Found 253 files belonging to 2 classes.
Using 50 files for validation.


In [ ]:
class_names = train_ds.class_names
print(class_names)

['no', 'yes']


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
num_classes = len(class_names)

In [ ]:
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(IMG_SIZE[0],
                                  IMG_SIZE[1],
                                  3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
  for i in range(9):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.axis("off")

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
        # Define the correct function signature for on_epoch_end
        def on_epoch_end(self, epochs, logs={}):
            if logs.get('accuracy') is not None and logs.get('accuracy') > 0.80 and logs.get('val_accuracy') > 0.85:
                print("\nReached 85% accuracy so cancelling training!")

                # Stop training once the above condition is met
                self.model.stop_training = True

In [ ]:
class My_Model(tf.keras.Model):
    def __init__(self):
        super(My_Model, self).__init__()
        self.data_augmentation = data_augmentation

        self.conv1 = layers.Conv2D(32, 3, padding='same', activation='relu')
        self.maxpool1 = layers.MaxPooling2D()

        self.conv2 = layers.Conv2D(64, 3, padding='same', activation='relu')
        self.maxpool2 = layers.MaxPooling2D()

        self.conv3 = layers.Conv2D(128, 3, padding='same', activation='relu')
        self.maxpool3 = layers.MaxPooling2D()

        self.dropout = layers.Dropout(0.2)
        self.flatten = layers.Flatten()
        self.dense1 = layers.Dense(128, activation='relu')
        self.dense2 = layers.Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        x = self.data_augmentation(inputs, training=training)
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.maxpool2(x)
        x = self.conv3(x)
        x = self.maxpool3(x)
        x = self.dropout(x)
        x = self.flatten(x)
        x = self.dense1(x)
        return self.dense2(x)

In [ ]:
model = My_Model()
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

In [ ]:
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs,
  callbacks=[myCallback()]
)

Epoch 1/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.5254 - loss: 510.9569 - val_accuracy: 0.6600 - val_loss: 1.6477
Epoch 2/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.6108 - loss: 1.0640 - val_accuracy: 0.7400 - val_loss: 0.5901
Epoch 3/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.7626 - loss: 0.5481 - val_accuracy: 0.7200 - val_loss: 0.6202
Epoch 4/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.8192 - loss: 0.4620 - val_accuracy: 0.7200 - val_loss: 0.5921
Epoch 5/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.7543 - loss: 0.5098 - val_accuracy: 0.7000 - val_loss: 0.6980
Epoch 6/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.7471 - loss: 0.5866 - val_accuracy: 0.7400 - val_loss: 0.5762
Epoch 7/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.7590 - loss: 0.4953 - val_accuracy: 0.7400 - val_loss: 0.7226
Epoch 8/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - accuracy: 0.7669 - loss: 0.5603 - val_accuracy: 0.7400 - val_loss: 0.6351
Epoch 

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# from keras.preprocessing import image

# nova_imagem = '/content/drive/MyDrive/val/angular_leaf_spot/angular_leaf_spot_val.42.jpg'

# img = image.load_img(nova_imagem, target_size=(180, 180))

# img_array = image.img_to_array(img)
# img_array = np.expand_dims(img_array, axis=0)

# previsao = model.predict(img_array)

# indice_classe_predita = np.argmax(previsao)

# rotulo_predito = "Angular Leaf Spot" if indice_classe_predita == 0 else "Bean Rust" if indice_classe_predita == 1 else "Healthy"

# print(f'Classe predita: {indice_classe_predita} - {rotulo_predito}')